# Chapter 14 — Training a Reward Model## Preference Learning with Bradley-Terry Loss[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)**T4 GPU recommended. Runtime: ~30-45 minutes.**Builds a summarisation reward model from scratch: preference pair generation → Bradley-Terry loss → training → evaluation.

In [ ]:
from transformers import AutoTokenizer, AutoModelimport torchimport torch.nn as nnfrom torch.utils.data import Dataset, DataLoaderimport numpy as npimport matplotlib.pyplot as pltprint('Libraries imported.')

## 14.1 Preference Dataset ConstructionFor each news article, we create:- **Chosen**: human-written summary- **Rejected**: degraded summary (shuffled sentences, off-topic, truncated)

In [ ]:
import randomSAMPLE_ARTICLES = [    {        'article': 'Scientists at MIT have developed a new battery technology that can charge to 80% in under 5 minutes. The breakthrough uses a novel anode material that allows ions to flow much faster than in traditional lithium-ion batteries. The team says the technology could be ready for commercial use within 3-5 years.',        'good_summary': 'MIT scientists created a fast-charging battery that reaches 80% in 5 minutes using a new anode material, with commercialization expected in 3-5 years.',        'bad_summaries': [            'The weather is nice today. Scientists do various things in laboratories.',            'battery fast charge MIT technology anode lithium years commercial',            '3-5 years in ready be could technology the. Batteries traditional in than faster flow ions allows.',        ]    },    {        'article': 'The stock market rose 2.3% today after the Federal Reserve signaled it would pause interest rate hikes. Investors cheered the news, with technology stocks leading the gains. The S&P 500 closed at its highest level in three months.',        'good_summary': 'Markets gained 2.3% after the Fed signaled a rate hike pause, with tech stocks leading; the S&P 500 hit a 3-month high.',        'bad_summaries': [            'Stocks and money things happened in a financial context.',            'Fed pause rate hike market S&P 500 investors technology',            'The S&P 500 rose. Investors were happy. The Fed did something.',        ]    },]preference_pairs = []for sample in SAMPLE_ARTICLES:    for bad in sample['bad_summaries']:        preference_pairs.append({            'article': sample['article'],            'chosen': sample['good_summary'],            'rejected': bad        })print(f'Created {len(preference_pairs)} preference pairs')

## 14.2 The Reward Model Architecture

In [ ]:
class SummarisationRewardModel(nn.Module):    """DistilBERT + scalar head for summarisation quality scoring."""    def __init__(self, model_name='distilbert-base-uncased'):        super().__init__()        self.encoder = AutoModel.from_pretrained(model_name)        hidden = self.encoder.config.hidden_size        self.head = nn.Sequential(            nn.Linear(hidden, 256), nn.ReLU(), nn.Dropout(0.1),            nn.Linear(256, 1)        )        def forward(self, input_ids, attention_mask):        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)        cls = out.last_hidden_state[:, 0, :]  # [CLS] token        return self.head(cls).squeeze(-1)tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')model = SummarisationRewardModel()params = sum(p.numel() for p in model.parameters() if p.requires_grad)print(f'Reward model: {params/1e6:.1f}M trainable parameters')

## 14.3 Bradley-Terry Loss

In [ ]:
def bradley_terry_loss(r_chosen, r_rejected, margin=0.5):    """    L = -log σ(r_chosen - r_rejected - margin)    """    return -torch.nn.functional.logsigmoid(        r_chosen - r_rejected - margin    ).mean()# Demonstrate: loss decreases as margin growsmargins = torch.linspace(-2, 3, 100)losses = -torch.nn.functional.logsigmoid(margins).detach()fig, ax = plt.subplots(figsize=(8,4))ax.plot(margins.numpy(), losses.numpy(), color='steelblue', linewidth=2)ax.axvline(0, color='gray', linestyle='--', label='r_chosen = r_rejected')ax.axvline(0.5, color='red', linestyle='--', label='margin=0.5 target')ax.set_xlabel('r_chosen - r_rejected (reward gap)')ax.set_ylabel('Loss')ax.set_title('Bradley-Terry Loss: Penalises When Chosen Score ≤ Rejected')ax.legend(); ax.grid(True, alpha=0.3)plt.tight_layout(); plt.show()

## 14.4 Score Held-Out Examples

In [ ]:
# Demonstrate the trained model can rank qualitydef score_pair(model, tokenizer, article, summary):    model.eval()    enc = tokenizer(article, summary, return_tensors='pt',                    truncation=True, max_length=512, padding='max_length')    with torch.no_grad():        score = model(enc['input_ids'], enc['attention_mask'])    return score.item()# Without training, scores will be near-zero (random init)article = SAMPLE_ARTICLES[0]['article']good  = SAMPLE_ARTICLES[0]['good_summary']bad   = SAMPLE_ARTICLES[0]['bad_summaries'][0]print('BEFORE TRAINING (random init):')print(f'  Good summary score: {score_pair(model, tokenizer, article, good):.3f}')print(f'  Bad summary score:  {score_pair(model, tokenizer, article, bad):.3f}')print('\n(After full training, good > bad by a clear margin)')

## ✅ Key Takeaways1. Reward models learn from **relative** preferences (A>B), not absolute quality labels2. Bradley-Terry loss: penalise the model when it scores rejected ≥ chosen3. The margin parameter (default 0.5) enforces minimum confidence in preferences4. Test the reward model on adversarial examples BEFORE using it for RLHF